# Pedestrian Detection & Instance Segmentation (YOLOv8-seg + ONNX)

End-to-end pipeline on the **Penn-Fudan Pedestrian** dataset:

1. Data preparation (PNG instance masks -> YOLO polygon labels, train/val split)
2. Zero-shot baseline (pretrained COCO weights, no training)
3. Three training experiments
4. Evaluation (box + mask metrics) and model selection
5. Optimization: ONNX export + CPU latency / size / accuracy comparison
6. Error analysis on the validation set
7. Gradio demo (runs the ONNX model)

**Run on Google Colab:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.
All outputs are written to `models/` and `results/`.

In [ ]:
!pip install -q ultralytics gradio onnx onnxruntime pandas matplotlib tabulate

In [ ]:
import glob
import os
import shutil
import time
import urllib.request
import zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image

SEED = 42        # reproducible split + training
EPOCHS = 40
IMGSZ = 640
BATCH = 8
VAL_RATIO = 0.2

np.random.seed(SEED)
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

## 1. Data preparation

Penn-Fudan has ~170 images. Each image has a mask PNG where **every pedestrian has its own pixel value**
(1, 2, 3, ...; 0 is background). YOLO segmentation labels need **one line per pedestrian**:
`class x1 y1 x2 y2 ...` with polygon coordinates normalised to 0..1.

In [ ]:
URL = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"
ZIP_PATH = "PennFudanPed.zip"

if not os.path.isdir("PennFudanPed"):
    if not os.path.exists(ZIP_PATH):
        print("Downloading Penn-Fudan Pedestrian dataset ...")
        urllib.request.urlretrieve(URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(".")
print("Dataset ready.")

img_paths = sorted(glob.glob("PennFudanPed/PNGImages/*.png"))
assert len(img_paths) > 0, "No images found - check the dataset download / extraction"
print(f"{len(img_paths)} images found")


def mask_path_for(img_path):
    # FudanPed00001.png  ->  FudanPed00001_mask.png  (paired by name, not by sort order)
    return f"PennFudanPed/PedMasks/{Path(img_path).stem}_mask.png"


MIN_AREA = 30  # ignore tiny noisy blobs (pixels)


def mask_to_yolo_polygons(mask_path):
    """One YOLO-seg line per pedestrian: 'class x1 y1 x2 y2 ...' (normalised)."""
    mask = np.array(Image.open(mask_path))
    h, w = mask.shape[:2]
    lines = []
    for obj_id in np.unique(mask):
        if obj_id == 0:  # background
            continue
        binary = (mask == obj_id).astype(np.uint8)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        contour = max(contours, key=cv2.contourArea)  # largest visible part of this person
        if len(contour) < 3 or cv2.contourArea(contour) < MIN_AREA:
            continue
        pts = contour.reshape(-1, 2).astype(np.float64)
        pts[:, 0] /= w
        pts[:, 1] /= h
        lines.append("0 " + " ".join(f"{v:.6f}" for v in pts.flatten()))
    return lines

In [ ]:
# Rebuild the YOLO dataset from scratch (avoids stale files from earlier runs)
if os.path.isdir("dataset"):
    shutil.rmtree("dataset")
for split in ["train", "val"]:
    os.makedirs(f"dataset/images/{split}")
    os.makedirs(f"dataset/labels/{split}")

perm = np.random.permutation(len(img_paths))
n_train = int(len(img_paths) * (1 - VAL_RATIO))
splits = {"train": perm[:n_train], "val": perm[n_train:]}

for split, idxs in splits.items():
    n_inst = 0
    for i in idxs:
        img_p = img_paths[i]
        base = Path(img_p).stem
        lines = mask_to_yolo_polygons(mask_path_for(img_p))
        shutil.copy(img_p, f"dataset/images/{split}/{base}.png")
        with open(f"dataset/labels/{split}/{base}.txt", "w") as f:
            f.write("\n".join(lines))
        n_inst += len(lines)
    print(f"{split:5s}: {len(idxs)} images, {n_inst} pedestrian instances")

with open("data.yaml", "w") as f:
    f.write(
        f"path: {os.path.abspath('dataset')}\n"
        "train: images/train\n"
        "val: images/val\n\n"
        "names:\n  0: person\n"
    )
print("data.yaml written")

## 2. Zero-shot baseline

The YOLOv8 weights are already pretrained on COCO, which **already contains a `person` class**.
So the honest question is: *does training on Penn-Fudan improve anything?*
We first measure the pretrained models on our validation split without any training.

In [ ]:
from ultralytics import YOLO


def collect_metrics(model, label):
    m = model.val(data="data.yaml", split="val", imgsz=IMGSZ, plots=False, verbose=False)
    return {
        "model": label,
        "box_P": m.box.mp, "box_R": m.box.mr, "box_mAP50": m.box.map50, "box_mAP50-95": m.box.map,
        "mask_P": m.seg.mp, "mask_R": m.seg.mr, "mask_mAP50": m.seg.map50, "mask_mAP50-95": m.seg.map,
    }


def model_info(weights_path):
    mdl = YOLO(str(weights_path))
    n_params = sum(p.numel() for p in mdl.model.parameters())
    return n_params / 1e6, os.path.getsize(weights_path) / 1e6


rows = []
for w in ["yolov8n-seg.pt", "yolov8s-seg.pt"]:
    try:
        row = collect_metrics(YOLO(w), f"zero-shot {w[:-3]}")
        row["params_M"], row["size_MB"] = model_info(w)
        row["train_min"] = np.nan
        rows.append(row)
    except Exception as e:  # keep the notebook running even if the baseline eval fails
        print(f"Baseline for {w} failed: {e}")
pd.DataFrame(rows).round(4)

## 3. Experiments

| Experiment | Start weights | What changes | Hypothesis |
|---|---|---|---|
| exp1_n_seg | yolov8n-seg | baseline fine-tune, all layers trained | small + fast, probably enough for one class |
| exp2_s_seg | yolov8s-seg | bigger model | more capacity -> better masks, but slower/larger |
| exp3_s_seg_freeze | yolov8s-seg | first 10 layers (backbone) frozen | with only ~136 training images, freezing may reduce overfitting |

Everything else (epochs, image size, batch, seed, augmentation defaults) is identical, so differences come from the change above.

In [ ]:
EXPERIMENTS = [
    # name,               start weights,    extra train args
    ("exp1_n_seg",        "yolov8n-seg.pt", {}),
    ("exp2_s_seg",        "yolov8s-seg.pt", {}),
    ("exp3_s_seg_freeze", "yolov8s-seg.pt", {"freeze": 10}),
]

trained = {}        # experiment name -> path to best.pt
train_minutes = {}  # experiment name -> wall-clock minutes

for name, weights, extra in EXPERIMENTS:
    print(f"\n=== {name}: {weights} {extra} ===")
    model = YOLO(weights)
    t0 = time.time()
    model.train(
        data="data.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED,
        project=os.path.abspath("experiments"), name=name, exist_ok=True, **extra,
    )
    train_minutes[name] = (time.time() - t0) / 60
    save_dir = Path(model.trainer.save_dir)
    trained[name] = save_dir / "weights" / "best.pt"
    assert trained[name].exists(), f"best.pt not found for {name}"
    if (save_dir / "results.png").exists():
        shutil.copy(save_dir / "results.png", f"results/{name}_curves.png")
    print(f"{name}: best weights -> {trained[name]}  ({train_minutes[name]:.1f} min)")

## 4. Evaluation and model selection

Metrics are computed on the held-out validation split, for both bounding boxes (**box**) and pixel masks (**mask**):
precision (P), recall (R), mAP@0.5 and mAP@0.5:0.95. The mask mAP@0.5:0.95 is the main metric
because the task is instance *segmentation*.

In [ ]:
for name, path in trained.items():
    row = collect_metrics(YOLO(str(path)), name)
    row["params_M"], row["size_MB"] = model_info(path)
    row["train_min"] = train_minutes[name]
    rows.append(row)

results = pd.DataFrame(rows).set_index("model").round(4)
results.to_csv("results/metrics.csv")
results

In [ ]:
import matplotlib.pyplot as plt

best_name = results.loc[list(trained), "mask_mAP50-95"].idxmax()
shutil.copy(trained[best_name], "models/best.pt")
print(f"Selected model: {best_name}  (highest mask mAP50-95 on validation)")

ax = results["mask_mAP50-95"].plot.barh(figsize=(8, 3.5), color="#4c72b0")
ax.set_xlabel("mask mAP50-95 (validation)")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("results/model_comparison.png", dpi=150)
plt.show()

## 5. Optimization: ONNX export

We export the selected model to ONNX and compare it with the PyTorch weights on the same machine, on CPU:
file size, end-to-end latency per image (pre-processing + inference + post-processing), and accuracy
(to verify that the export did not change the results).

In [ ]:
exported = YOLO("models/best.pt").export(format="onnx", imgsz=IMGSZ, simplify=True)
if os.path.abspath(exported) != os.path.abspath("models/best.onnx"):
    shutil.copy(exported, "models/best.onnx")  # export normally already writes it next to best.pt
print("ONNX model saved to models/best.onnx")

val_imgs = sorted(glob.glob("dataset/images/val/*.png"))[:20]


def bench(model, imgs, warmup=3, **kw):
    for p in imgs[:warmup]:
        model.predict(p, verbose=False, imgsz=IMGSZ, **kw)
    t0 = time.perf_counter()
    for p in imgs:
        model.predict(p, verbose=False, imgsz=IMGSZ, **kw)
    return (time.perf_counter() - t0) / len(imgs) * 1000  # ms / image


pt_model = YOLO("models/best.pt")
onnx_model = YOLO("models/best.onnx", task="segment")

pt_ms = bench(pt_model, val_imgs, device="cpu")
onnx_ms = bench(onnx_model, val_imgs)

try:
    onnx_metrics = collect_metrics(onnx_model, "onnx")
    onnx_map = onnx_metrics["mask_mAP50-95"]
except Exception as e:
    print(f"ONNX validation failed: {e}")
    onnx_map = float("nan")

optimization = pd.DataFrame([
    {"format": "PyTorch (.pt)", "size_MB": os.path.getsize("models/best.pt") / 1e6,
     "cpu_ms_per_image": pt_ms, "mask_mAP50-95": results.loc[best_name, "mask_mAP50-95"]},
    {"format": "ONNX (.onnx)", "size_MB": os.path.getsize("models/best.onnx") / 1e6,
     "cpu_ms_per_image": onnx_ms, "mask_mAP50-95": onnx_map},
]).set_index("format").round(4)
optimization.to_csv("results/optimization.csv")
print(f"Speed-up on CPU: {pt_ms / onnx_ms:.2f}x")
optimization

## 6. Error analysis

For every validation image we compare the number of ground-truth pedestrians with the number detected
(conf >= 0.25). This is a simple count-based proxy, not an IoU-based analysis, but it shows *where* the model fails.
**Look at the saved images and describe what you actually see in the technical report.**

In [ ]:
best_model = YOLO("models/best.pt")
err_rows = []
for img_p in sorted(glob.glob("dataset/images/val/*.png")):
    base = Path(img_p).stem
    with open(f"dataset/labels/val/{base}.txt") as f:
        gt = sum(1 for line in f if line.strip())
    r = best_model.predict(img_p, conf=0.25, imgsz=IMGSZ, verbose=False)[0]
    pred = 0 if r.boxes is None else len(r.boxes)
    err_rows.append({"image": base, "gt": gt, "pred": pred, "diff": pred - gt})

err = pd.DataFrame(err_rows)
print(f"Images with the correct pedestrian count: {(err['diff'] == 0).sum()} / {len(err)}")
print(f"Missed pedestrians (pred < gt): {-err.loc[err['diff'] < 0, 'diff'].sum()}")
print(f"Extra detections   (pred > gt): {err.loc[err['diff'] > 0, 'diff'].sum()}")
err.to_csv("results/error_analysis.csv", index=False)

worst = err.loc[err["diff"].abs().sort_values(ascending=False).index]
worst = worst[worst["diff"] != 0].head(6)
if len(worst):
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax in axes.flat:
        ax.axis("off")
    for ax, (_, row) in zip(axes.flat, worst.iterrows()):
        path = f"dataset/images/val/{row['image']}.png"
        r = best_model.predict(path, conf=0.25, imgsz=IMGSZ, verbose=False)[0]
        ax.imshow(r.plot()[..., ::-1])  # BGR -> RGB
        ax.set_title(f"{row['image']}  GT={row['gt']}  predicted={row['pred']}")
    plt.tight_layout()
    plt.savefig("results/error_analysis.png", dpi=120)
    plt.show()
else:
    print("No count mismatches on the validation set.")

In [ ]:
# Markdown tables that can be pasted straight into TECHNICAL_REPORT.md
with open("results/tables.md", "w") as f:
    f.write("### Experiment results (validation split)\n\n")
    f.write(results.to_markdown() + "\n\n")
    f.write("### Optimization: PyTorch vs ONNX (CPU)\n\n")
    f.write(optimization.to_markdown() + "\n")
print(open("results/tables.md").read())

## 7. Demo (Gradio, runs the ONNX model)

`app.py` is written to disk so the same file can be run outside Colab (`python app.py`).

In [ ]:
%%writefile app.py
"""Gradio demo: pedestrian detection + instance segmentation.

Runs the exported ONNX model (models/best.onnx) through Ultralytics /
ONNX Runtime. Falls back to the PyTorch weights (models/best.pt) if the
ONNX file is missing.

Usage:
    python app.py                       # local, http://127.0.0.1:7860
    MODEL_PATH=models/best.pt python app.py
"""
import os

import gradio as gr
from ultralytics import YOLO

IMGSZ = 640  # must match the size used when exporting the ONNX model


def _load_model():
    path = os.environ.get("MODEL_PATH", "models/best.onnx")
    if not os.path.exists(path):
        fallback = "models/best.pt"
        if not os.path.exists(fallback):
            raise FileNotFoundError(
                f"Model not found: {path} (and no {fallback}). "
                "Run main.ipynb first, or download the weights - see README."
            )
        print(f"[app] {path} not found, using {fallback}")
        path = fallback
    print(f"[app] loading model: {path}")
    return YOLO(path, task="segment")


model = _load_model()


def predict(image, conf):
    """image: PIL image -> (annotated RGB image, text summary)."""
    if image is None:
        return None, "No image."
    result = model.predict(image, conf=float(conf), imgsz=IMGSZ, verbose=False)[0]
    n = 0 if result.boxes is None else len(result.boxes)
    annotated_bgr = result.plot()  # Ultralytics returns BGR; Gradio expects RGB
    return annotated_bgr[..., ::-1], f"Detected pedestrians: {n}"


def _conf_slider():
    return gr.Slider(0.05, 0.95, value=0.25, step=0.05, label="Confidence threshold")


def _outputs():
    return [gr.Image(type="numpy", label="Segmentation result"), gr.Textbox(label="Summary")]


def build_demo():
    upload = gr.Interface(
        fn=predict,
        inputs=[gr.Image(sources=["upload"], type="pil", label="Upload image"), _conf_slider()],
        outputs=_outputs(),
        title="Upload image",
    )
    snapshot = gr.Interface(
        fn=predict,
        inputs=[gr.Image(sources=["webcam"], type="pil", label="Webcam snapshot"), _conf_slider()],
        outputs=_outputs(),
        title="Webcam snapshot",
    )
    live = gr.Interface(
        fn=predict,
        inputs=[
            gr.Image(sources=["webcam"], type="pil", streaming=True, label="Live webcam"),
            _conf_slider(),
        ],
        outputs=_outputs(),
        live=True,
        title="Live stream",
    )
    return gr.TabbedInterface(
        [upload, snapshot, live],
        ["1. Upload image", "2. Webcam snapshot", "3. Live stream"],
        title="Pedestrian Detection & Instance Segmentation (YOLOv8-seg + ONNX)",
    )


if __name__ == "__main__":
    build_demo().launch()

In [ ]:
from app import build_demo

demo = build_demo()
demo.launch(share=True)  # public link, valid while this Colab session is running

## 8. Package the outputs

Download `deliverables.zip` and commit its contents (`models/`, `results/`, `app.py`) to the GitHub repository.

In [ ]:
!zip -rq deliverables.zip models results app.py data.yaml
try:
    from google.colab import files
    files.download("deliverables.zip")
except ImportError:
    print("deliverables.zip created")